# Lesson 08: ETL Pipeline

**Extract → Transform → Load** from OLTP to Star Schema

In this notebook, you'll build an ETL pipeline that:
1. **Extract** data from the OLTP source (FreeSQL)
2. **Transform** it using pandas — clean, join, aggregate
3. **Load** it into the star schema data warehouse

---

**Before you start:**
- Run `01_setup_oltp.sql` and `02_setup_dw.sql` in FreeSQL
- Replace `PASSWORD` below with your FreeSQL password

In [31]:
# === SETUP ===
# Install Oracle driver (Colab doesn't have it by default)
!pip install oracledb pandas sqlalchemy plotly --quiet

import oracledb
import pandas as pd
import plotly.express as px
from sqlalchemy import create_engine
import datetime
from datetime import date, timedelta

print('Libraries loaded successfully')


Libraries loaded successfully


In [48]:
# === CONNECT TO FREESQL ===
# Replace with YOUR FreeSQL credentials
# Get these from https://freesql.com/ -> Connection Details

USERNAME = ""
PASSWORD = ""  # <-- REPLACE THIS
DSN = "tcps://db.freesql.com:2484/23ai_34ui2"


# Create SQLAlchemy engine (used by pandas read_sql)
engine = create_engine(
    "oracle+oracledb://:@",
    connect_args={
        "user": USERNAME,
        "password": PASSWORD,
        "dsn": DSN
    }
)

# Quick test
df_test = pd.read_sql("SELECT COUNT(*) AS task_count FROM tasks", engine)
print(f"Connected! Tasks in database: {df_test.iloc[0]['task_count']}")


Connected! Tasks in database: 40


In [38]:
# Quick test
df_test = pd.read_sql("SELECT 'dim_status: ' || COUNT(*) FROM dim_status", engine)
print(f"Connected! Tasks in database: {df_test}")


Connected! Tasks in database:   'DIM_STATUS:'||COUNT(*)
0           dim_status: 5


---
## Phase 1: EXTRACT

Read data from the OLTP source tables. This is the simplest step — we just SELECT everything.

In [39]:
# === EXTRACT: Read source tables ===

users_df = pd.read_sql("SELECT * FROM users ORDER BY id", engine)
tasks_df = pd.read_sql("SELECT * FROM tasks ORDER BY id", engine)
assignments_df = pd.read_sql("SELECT * FROM task_assignments ORDER BY task_id, valid_from", engine)

print(f"Extracted {len(users_df)} users")
print(f"Extracted {len(tasks_df)} tasks")
print(f"Extracted {len(assignments_df)} assignment records")
print()
print('=== Users (first 3) ===')
display(users_df.head(3))
print()
print('=== Tasks (first 3) ===')
display(tasks_df.head(3))
print()
print('=== Assignments (first 5) ===')
display(assignments_df.head(5))




Extracted 8 users
Extracted 40 tasks
Extracted 41 assignment records

=== Users (first 3) ===


,id,name,email,team,role,created_at
0,1,Alice Chen,alice@example.com,Platform,senior,2026-05-26 13:33:22.964665
1,2,Bob Martinez,bob@example.com,Platform,developer,2026-05-26 13:33:22.968652
2,3,Carol Smith,carol@example.com,Frontend,senior,2026-05-26 13:33:22.970849



=== Tasks (first 3) ===


,id,title,description,status,priority,assigned_to,created_by,created_at,updated_at,completed_at
0,1,Fix login redirect bug,Users redirected to home instead of dashboard ...,completed,high,1,1,2026-02-01 09:00:00,2026-02-02 14:00:00,2026-02-02 14:00:00
1,2,Design new dashboard mockups,Create Figma mockups for analytics dashboard w...,completed,medium,3,3,2026-02-03 10:00:00,2026-02-10 16:00:00,2026-02-10 16:00:00
2,3,Upgrade numpy and pandas,"Update to latest stable versions, fix breaking...",completed,low,5,5,2026-02-05 11:00:00,2026-02-07 15:00:00,2026-02-07 15:00:00



=== Assignments (first 5) ===


,assignment_id,task_id,assigned_to,assigned_by,valid_from,valid_to
0,1,1,1,1.0,2026-02-01 09:00:00,NaT
1,2,2,3,3.0,2026-02-03 10:00:00,NaT
2,3,3,5,5.0,2026-02-05 11:00:00,NaT
3,4,4,1,2.0,2026-02-06 09:00:00,NaT
4,5,5,2,1.0,2026-02-07 10:00:00,2026-02-10 10:00:00


**Pause and think:**
- Why is the `tasks` table hard to query for analytics?
- What would a query like "tasks completed per team per month" look like on this schema?

The answer: you'd need to JOIN users + tasks, GROUP BY team + month, filter by status='completed'. Multiple joins, full table scans. That's fine for one query — but when every dashboard widget needs a different slice, it becomes slow.

---
## Phase 2: TRANSFORM

This is where the real work happens. We'll:
1. Build a **date dimension** covering the full date range
2. Join users and tasks to create a denormalized view
3. Aggregate by day, user, status, and priority
4. Compute metrics: tasks created, tasks completed, avg completion time

In [40]:
# === TRANSFORM Step 1: Build Date Dimension ===

# Find the date range from the tasks data
min_date = tasks_df['created_at'].min().date()
max_date = date.today()
print(f"Date range: {min_date} to {max_date}")

# Generate every day in the range
date_rows = []
current = min_date
while current <= max_date:
    date_rows.append({
        'date_key': int(current.strftime('%Y%m%d')),
        'full_date': current,
        'year': current.year,
        'quarter': (current.month - 1) // 3 + 1,
        'month': current.month,
        'month_name': current.strftime('%B'),
        'day': current.day,
        'day_name': current.strftime('%A'),
        'is_weekend': 1 if current.weekday() >= 5 else 0
    })
    current += timedelta(days=1)

dim_date_df = pd.DataFrame(date_rows)
print(f"Generated {len(dim_date_df)} days in date dimension")
display(dim_date_df.head(5))
display(dim_date_df.tail(3))

Date range: 2026-02-01 to 2026-05-26
Generated 115 days in date dimension


,date_key,full_date,year,quarter,month,month_name,day,day_name,is_weekend
0,20260201,2026-02-01,2026,1,2,February,1,Sunday,1
1,20260202,2026-02-02,2026,1,2,February,2,Monday,0
2,20260203,2026-02-03,2026,1,2,February,3,Tuesday,0
3,20260204,2026-02-04,2026,1,2,February,4,Wednesday,0
4,20260205,2026-02-05,2026,1,2,February,5,Thursday,0


,date_key,full_date,year,quarter,month,month_name,day,day_name,is_weekend
112,20260524,2026-05-24,2026,2,5,May,24,Sunday,1
113,20260525,2026-05-25,2026,2,5,May,25,Monday,0
114,20260526,2026-05-26,2026,2,5,May,26,Tuesday,0


In [41]:
# === TRANSFORM Step 2: Build User Dimension ===

dim_user_df = users_df.rename(columns={'id': 'user_id'}).copy()
dim_user_df = dim_user_df[['user_id', 'name', 'email', 'team', 'role']]
print(f"User dimension: {len(dim_user_df)} rows")
display(dim_user_df.head())

User dimension: 8 rows


,user_id,name,email,team,role
0,1,Alice Chen,alice@example.com,Platform,senior
1,2,Bob Martinez,bob@example.com,Platform,developer
2,3,Carol Smith,carol@example.com,Frontend,senior
3,4,Dave Kim,dave@example.com,Frontend,developer
4,5,Eve Johnson,eve@example.com,Data,senior


In [42]:
# === TRANSFORM Step 3: Build Status Dimension ===
# We already seeded this in 02_setup_dw.sql, but let's build it here too
# for reference — shows the dimension is just a lookup table

status_data = [
    ('open', 'active'),
    ('in_progress', 'active'),
    ('blocked', 'active'),
    ('completed', 'done'),
    ('cancelled', 'cancelled'),
]
dim_status_df = pd.DataFrame(status_data, columns=['status_name', 'category'])
print('Status dimension:')
display(dim_status_df)

Status dimension:


,status_name,category
0,open,active
1,in_progress,active
2,blocked,active
3,completed,done
4,cancelled,cancelled


In [43]:
# === TRANSFORM Step 4: Compute completion time ===
# For completed tasks, calculate hours between created_at and completed_at

tasks_with_completion = tasks_df[
    (tasks_df['status'] == 'completed') &
    (tasks_df['completed_at'].notna())
].copy()

tasks_with_completion['completion_hours'] = (
    (tasks_with_completion['completed_at'] - tasks_with_completion['created_at'])
).dt.total_seconds() / 3600

print(f"Completed tasks with timing data: {len(tasks_with_completion)}")
display(tasks_with_completion[['title', 'created_at', 'completed_at', 'completion_hours']].head())

Completed tasks with timing data: 25


,title,created_at,completed_at,completion_hours
0,Fix login redirect bug,2026-02-01 09:00:00,2026-02-02 14:00:00,29.0
1,Design new dashboard mockups,2026-02-03 10:00:00,2026-02-10 16:00:00,174.0
2,Upgrade numpy and pandas,2026-02-05 11:00:00,2026-02-07 15:00:00,52.0
3,API rate limiting,2026-02-06 09:00:00,2026-02-12 11:00:00,146.0
4,Write unit tests for auth,2026-02-07 10:00:00,2026-02-14 17:00:00,175.0


In [45]:
# === TRANSFORM Step 5: Build the Fact Table ===
# This is the core transformation: aggregate by (date, user, status, priority)
#
# KEY INSIGHT: We use task_assignments (not tasks.assigned_to) to determine
# who was assigned at each point in time. This handles reassignments correctly:
# - A task created by Alice (assigned to Bob at creation) -> Bob gets creation credit
# - A task reassigned to Grace and completed by Grace -> Grace gets completion credit

print(f"Assignment records available: {len(assignments_df)}")

# --- Tasks Created: who was assigned at creation time? ---
# Merge tasks with assignments, then filter to the assignment active at CREATED_AT
created_assign = tasks_df[['id', 'created_at', 'status', 'priority']].merge(
    assignments_df, left_on='id', right_on='task_id', how='inner'
)
created_assign = created_assign[
    (created_assign['created_at'] >= created_assign['valid_from']) &
    (created_assign['valid_to'].isna() | (created_assign['created_at'] < created_assign['valid_to']))
].copy()
created_assign['created_date'] = created_assign['created_at'].dt.date

created_by_day = created_assign.groupby(
    ['created_date', 'assigned_to', 'status', 'priority']
).size().reset_index(name='tasks_created')

# --- Tasks Completed: who was assigned at completion time? ---
completed_tasks = tasks_with_completion[['id', 'completed_at', 'status', 'priority', 'completion_hours']].merge(
    assignments_df, left_on='id', right_on='task_id', how='inner'
)
completed_tasks = completed_tasks[
    (completed_tasks['completed_at'] >= completed_tasks['valid_from']) &
    (completed_tasks['valid_to'].isna() | (completed_tasks['completed_at'] < completed_tasks['valid_to']))
].copy()
completed_tasks['completed_date'] = completed_tasks['completed_at'].dt.date

completed_by_day = completed_tasks.groupby(
    ['completed_date', 'assigned_to', 'status', 'priority']
).size().reset_index(name='tasks_completed')

# Average completion time per (user, priority)
# Uses the assignee at completion time for accuracy
avg_completion = completed_tasks.groupby(
    ['assigned_to', 'priority']
)['completion_hours'].mean().reset_index(name='avg_completion_hours')

print(f"Created-by-day rows: {len(created_by_day)}")
print(f"Completed-by-day rows: {len(completed_by_day)}")
print(f"Avg completion rows: {len(avg_completion)}")
display(created_by_day.head())

Assignment records available: 41
Created-by-day rows: 40
Completed-by-day rows: 25
Avg completion rows: 14


,created_date,assigned_to,status,priority,tasks_created
0,2026-02-01,1,completed,high,1
1,2026-02-03,3,completed,medium,1
2,2026-02-05,5,completed,low,1
3,2026-02-06,1,completed,high,1
4,2026-02-07,2,completed,medium,1


In [46]:
# === TRANSFORM Step 6: Merge into one fact table ===
#
# NOTE: ASSIGNED_TO now reflects the HISTORICAL assignee from task_assignments,
# not the current assigned_to from tasks. Reassignments are handled correctly.

# === TRANSFORM Step 6: Merge into one fact table ===

# Start with created data, left join completed data
fact = created_by_day.rename(columns={'created_date': 'date'})

completed_by_day_renamed = completed_by_day.rename(
    columns={'completed_date': 'date'}
)

fact = fact.merge(
    completed_by_day_renamed,
    on=['date', 'assigned_to', 'status', 'priority'],
    how='left'
)

# Add average completion time
fact = fact.merge(
    avg_completion,
    on=['assigned_to', 'priority'],
    how='left'
)

# Fill NaN values
fact['tasks_completed'] = fact['tasks_completed'].fillna(0).astype(int)

# Add date_key
fact['date_key'] = fact['date'].apply(lambda d: int(d.strftime('%Y%m%d')))

# Add user_key (we'll need to map this during load)
print(f"Fact table rows: {len(fact)}")
display(fact.head())
print(f"\nColumns: {list(fact.columns)}")

Fact table rows: 40


,date,assigned_to,status,priority,tasks_created,tasks_completed,avg_completion_hours,date_key
0,2026-02-01,1,completed,high,1,0,115.333333,20260201
1,2026-02-03,3,completed,medium,1,0,173.500000,20260203
2,2026-02-05,5,completed,low,1,0,52.000000,20260205
3,2026-02-06,1,completed,high,1,0,115.333333,20260206
4,2026-02-07,2,completed,medium,1,0,219.666667,20260207



Columns: ['date', 'assigned_to', 'status', 'priority', 'tasks_created', 'tasks_completed', 'avg_completion_hours', 'date_key']


---
## Phase 3: LOAD

Now we write the transformed data into the star schema tables in FreeSQL.

**The order matters:**
1. Load `dim_date` first (fact table references it)
2. Load `dim_user` first (fact table references it)
3. Load `fact_task_daily` last

In [49]:
# === LOAD Step 1: Insert into dim_date ===

date_insert_sql = """
    MERGE INTO dim_date d
    USING (SELECT :1 AS date_key, :2 AS full_date, :3 AS year,
                  :4 AS quarter, :5 AS month, :6 AS month_name,
                  :7 AS day, :8 AS day_name, :9 AS is_weekend FROM dual) src
    ON (d.date_key = src.date_key)
    WHEN NOT MATCHED THEN INSERT
        (date_key, full_date, year, quarter, month, month_name,
         day, day_name, is_weekend)
        VALUES (src.date_key, src.full_date, src.year, src.quarter,
                src.month, src.month_name, src.day, src.day_name,
                src.is_weekend)
"""

raw_connection = engine.raw_connection()
cursor = raw_connection.cursor()
rows_inserted = 0
for _, row in dim_date_df.iterrows():
    cursor.execute(date_insert_sql, [
        int(row['date_key']),
        row['full_date'],
        int(row['year']),
        int(row['quarter']),
        int(row['month']),
        row['month_name'],
        int(row['day']),
        row['day_name'],
        int(row['is_weekend'])
    ])
    rows_inserted += 1

raw_connection.commit()
print(f"Loaded {rows_inserted} rows into dim_date")


ERROR:sqlalchemy.pool.impl.QueuePool:Exception during reset or similar
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sqlalchemy/pool/base.py", line 986, in _finalize_fairy
    fairy._reset(
  File "/usr/local/lib/python3.12/dist-packages/sqlalchemy/pool/base.py", line 1441, in _reset
    pool._dialect.do_rollback(self)
  File "/usr/local/lib/python3.12/dist-packages/sqlalchemy/engine/default.py", line 712, in do_rollback
    dbapi_connection.rollback()
  File "/usr/local/lib/python3.12/dist-packages/oracledb/connection.py", line 1327, in rollback
    self._impl.rollback()
  File "src/oracledb/impl/thin/connection.pyx", line 662, in oracledb.thin_impl.ThinConnImpl.rollback
  File "src/oracledb/impl/thin/connection.pyx", line 171, in oracledb.thin_impl.BaseThinConnImpl._create_message
  File "src/oracledb/impl/thin/messages/base.pyx", line 113, in oracledb.thin_impl.Message._initialize
  File "src/oracledb/impl/thin/packet.pyx", line 235, in oracledb.

Loaded 115 rows into dim_date


In [50]:
# === LOAD Step 2: Insert into dim_user ===

user_insert_sql = """
    MERGE INTO dim_user u
    USING (SELECT :1 AS user_id, :2 AS name, :3 AS email,
                  :4 AS team, :5 AS role FROM dual) src
    ON (u.user_id = src.user_id)
    WHEN NOT MATCHED THEN INSERT
        (user_id, name, email, team, role)
        VALUES (src.user_id, src.name, src.email, src.team, src.role)
    WHEN MATCHED THEN UPDATE SET
        name = src.name, email = src.email,
        team = src.team, role = src.role
"""

rows_inserted = 0
for _, row in dim_user_df.iterrows():
    cursor.execute(user_insert_sql, [
        int(row['user_id']),
        row['name'],
        row['email'],
        row['team'],
        row['role']
    ])
    rows_inserted += 1

raw_connection.commit()
print(f"Loaded {rows_inserted} rows into dim_user")

Loaded 8 rows into dim_user


In [51]:
# === LOAD Step 3: Map user_key and status_key ===
# We need to look up the surrogate keys from the dimension tables
#
# NOTE: ASSIGNED_TO is the historical assignee from task_assignments.
# The mapping to dim_user works the same way.

# === LOAD Step 3: Map user_key and status_key ===
# We need to look up the surrogate keys from the dimension tables

# Read the dimension keys from the DW
user_key_map = pd.read_sql("SELECT user_key, user_id FROM dim_user", engine)
status_key_map = pd.read_sql("SELECT status_key, status_name FROM dim_status", engine)

# Join keys into the fact table
fact_with_keys = fact.merge(user_key_map, left_on='assigned_to', right_on='user_id')
fact_with_keys = fact_with_keys.merge(status_key_map, left_on='status', right_on='status_name')

print(f"Fact rows with keys: {len(fact_with_keys)}")
display(fact_with_keys[['date_key', 'user_key', 'status_key', 'priority', 'tasks_created', 'tasks_completed']].head())

Fact rows with keys: 40


,date_key,user_key,status_key,priority,tasks_created,tasks_completed
0,20260201,1,4,high,1,0
1,20260203,3,4,medium,1,0
2,20260205,5,4,low,1,0
3,20260206,1,4,high,1,0
4,20260207,2,4,medium,1,0


In [53]:
# === LOAD Step 4: Insert into fact_task_daily ===

fact_insert_sql = """
    MERGE INTO fact_task_daily f
    USING (SELECT :1 AS date_key, :2 AS user_key, :3 AS status_key,
                  :4 AS priority, :5 AS tasks_created, :6 AS tasks_completed,
                  :7 AS avg_completion_hours FROM dual) src
    ON (f.date_key = src.date_key AND f.user_key = src.user_key
        AND f.status_key = src.status_key AND f.priority = src.priority)
    WHEN NOT MATCHED THEN INSERT
        (date_key, user_key, status_key, priority, tasks_created, tasks_completed, avg_completion_hours)
        VALUES (src.date_key, src.user_key, src.status_key, src.priority,
                src.tasks_created, src.tasks_completed, src.avg_completion_hours)
    WHEN MATCHED THEN UPDATE SET
        tasks_created = src.tasks_created,
        tasks_completed = src.tasks_completed,
        avg_completion_hours = src.avg_completion_hours
"""

rows_inserted = 0
for _, row in fact_with_keys.iterrows():
    avg_hrs = row.get('avg_completion_hours')
    if pd.isna(avg_hrs):
        avg_hrs = None
    cursor.execute(fact_insert_sql, [
        int(row['date_key']),
        int(row['user_key']),
        int(row['status_key']),
        row['priority'],
        int(row['tasks_created']),
        int(row['tasks_completed']),
        avg_hrs
    ])
    rows_inserted += 1

raw_connection.commit()
print(f"Loaded {rows_inserted} rows into fact_task_daily")

Loaded 40 rows into fact_task_daily


In [54]:
# === VERIFY: Check what we loaded ===

verify = pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(tasks_created) AS total_created,
        SUM(tasks_completed) AS total_completed,
        ROUND(AVG(avg_completion_hours), 1) AS avg_completion_hours
    FROM fact_task_daily
""", engine)

display(verify)

# Compare with source
source_check = pd.read_sql("""
    SELECT
        COUNT(*) AS total_tasks,
        SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed_tasks
    FROM tasks
""", engine)

print("Source check:")
display(source_check)

,total_rows,total_created,total_completed,avg_completion_hours
0,40,40,1,156.3


Source check:


,total_tasks,completed_tasks
0,40,25


In [ ]:
# === CLEANUP ===
cursor.close()
raw_connection.close()
print('Connection closed. ETL pipeline complete!')

---
## What Just Happened?

You built a complete ETL pipeline:

| Step | What | Tool |
|------|------|------|
| **Extract** | Read users and tasks from FreeSQL | `pd.read_sql()` |
| **Transform** | Built date dimension, joined tables, aggregated by day/user/status | pandas |
| **Load** | Inserted into star schema with MERGE (upsert) | `cursor.execute()` |

**Why this matters:**
- The star schema is now optimized for analytical queries
- Queries that needed 3+ JOINs on the OLTP now need 1-2 JOINs
- The date dimension makes time-based filtering trivial
- The fact table is append-only — no UPDATEs, no locking contention

**Next step:** Run `04_analytical_queries.sql` in FreeSQL to see the payoff.